# Module 06 — Lesson 06
# Data Transformation (Skewness, Log Transform)

---

> `salary` and `years_experience` aren't just differently *scaled* from `age` (Lesson 05's problem) — they're differently *shaped*. Most employees cluster at the low end of experience and salary, with a long tail of a few people stretching far to the right. That shape, not just the units, is its own kind of messiness — and it's the last thing standing between this dataset and being fully analysis-ready.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Measure **skewness** with `.skew()` and read what the sign and size of the number mean
- Apply a **log transform** (`np.log`, `np.log1p`) to pull in a long right tail
- Apply a **square root transform** as a gentler alternative for moderate skew
- Explain why `np.log1p` (not plain `np.log`) is required for columns that contain zero
- Reverse a transform (`np.expm1`) to bring a value back to its original, interpretable units
- Place transformation correctly in a cleaning pipeline: **before** scaling, not after

In [1]:
import numpy as np
import pandas as pd

employees = pd.read_csv("data/employees_clean.csv")
print(f"employees.shape: {employees.shape}")
employees[["employee_id", "age", "years_experience", "salary"]].describe()

employees.shape: (49, 7)


,employee_id,age,years_experience,salary
count,49.000000,49.000000,49.000000,49.000000
mean,1030.061224,38.183673,4.502041,73361.224490
std,18.201887,9.801517,4.358253,28040.237579
min,1001.000000,22.000000,0.000000,47100.000000
25%,1015.000000,28.000000,1.400000,58000.000000
50%,1028.000000,38.000000,3.100000,71800.000000
75%,1046.000000,45.000000,5.700000,79800.000000
max,1060.000000,57.000000,22.000000,245000.000000


---
## 1. Measuring Skewness

`.skew()` gives a single number summarizing a distribution's lopsidedness. As a rule of thumb: values near 0 mean roughly symmetric, positive values mean a long tail stretching to the **right** (a few unusually high values), and negative values mean a long tail to the **left**. You don't need a plot to notice a hint of this either — compare the mean to the median in `.describe()` above: when the mean sits well above the median, high outliers are pulling it up, which is exactly what a right skew looks like.

In [2]:
skewness = employees[["age", "years_experience", "salary"]].skew()
print("Skewness per column:")
print(skewness.round(3))

Skewness per column:
age                 0.119
years_experience    1.849
salary              4.902
dtype: float64


`age` is close to 0 — roughly symmetric, no transform needed. `years_experience` is moderately right-skewed (most employees are early-career, a handful are veterans). `salary` is dramatically right-skewed — no surprise, since Lesson 03's genuine 245,000 outlier is still sitting in this column, exactly where it belongs.

---
## 2. The Log Transform

`np.log()` compresses large values much more aggressively than small ones, which pulls in a long right tail and makes the distribution closer to symmetric. It's the standard first move for right-skewed data like income, population, or counts.

In [3]:
salary_log = np.log(employees["salary"])
print(f"salary skew before log: {employees['salary'].skew():.3f}")
print(f"salary skew after log:  {salary_log.skew():.3f}")

salary skew before log: 4.902
salary skew after log:  2.176


Skew dropped a lot, but not all the way to 0 — and that's the right outcome, not a failure. A log transform shrinks how *extreme* the 245,000 outlier looks relative to everyone else, but it can't erase the fact that this employee's salary really is unusually high. That's consistent with Lesson 03: this value is genuine, not an error, so no amount of mathematical massaging should make it disappear entirely.

---
## 3. Zeros Break Plain `log` — Use `log1p`

`years_experience` contains employees with **0** years of experience (new hires). `np.log(0)` is undefined (negative infinity) — try it on the raw column and pandas will produce `-inf`, which silently poisons every calculation downstream. `np.log1p(x)` computes `log(1 + x)` instead, which is perfectly safe at `x = 0`.

In [4]:
print("np.log(0) breaks down:", np.log(employees["years_experience"]).replace([np.inf, -np.inf], np.nan).isna().sum(),
      "undefined values produced by plain log on a column containing zero")

years_experience_log = np.log1p(employees["years_experience"])
print(f"years_experience skew before: {employees['years_experience'].skew():.3f}")
print(f"years_experience skew after log1p: {years_experience_log.skew():.3f}")

np.log(0) breaks down: 1 undefined values produced by plain log on a column containing zero
years_experience skew before: 1.849
years_experience skew after log1p: 0.168


/Users/aminuddin/Desktop/YTV/data-science/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


From 1.85 down to 0.17 — `years_experience` is now close to symmetric. Unlike `salary`, this column doesn't have a single dominant outlier; it's a naturally exponential-shaped distribution (lots of junior employees, progressively fewer senior ones), which is exactly the shape a log transform is best at fixing.

---
## 4. A Gentler Alternative: Square Root

When skew is moderate rather than extreme, `np.sqrt()` pulls in the tail less aggressively than `log`. It also handles 0 natively (no `log1p`-style trick needed), but like `log`, it's undefined for negative values.

In [5]:
years_experience_sqrt = np.sqrt(employees["years_experience"])
salary_sqrt = np.sqrt(employees["salary"])

comparison = pd.DataFrame({
    "column": ["years_experience", "salary"],
    "original_skew": [employees["years_experience"].skew(), employees["salary"].skew()],
    "sqrt_skew": [years_experience_sqrt.skew(), salary_sqrt.skew()],
    "log_skew": [years_experience_log.skew(), salary_log.skew()],
}).round(3)
comparison

,column,original_skew,sqrt_skew,log_skew
0,years_experience,1.849,0.622,0.168
1,salary,4.902,3.561,2.176


For both columns here, `log`/`log1p` reduces skew further than `sqrt` — that tracks with how strong the original skew was. As a rule of thumb: reach for `sqrt` on mild skew, and `log`/`log1p` on strong skew like this dataset's.

---
## 5. Reversing a Transform

A model trained on log-transformed `years_experience` will make predictions *in log units* — not directly readable as "years." `np.expm1()` (the exact inverse of `log1p`) converts back to the original scale. This matters the moment you want to report or interpret a result in Module 10 and beyond.

In [6]:
round_trip = np.expm1(years_experience_log)

print("Original vs. log1p -> expm1 round trip (should match, aside from tiny floating-point noise):")
pd.DataFrame({
    "original": employees["years_experience"].head(),
    "round_trip": round_trip.head().round(6)
})

Original vs. log1p -> expm1 round trip (should match, aside from tiny floating-point noise):


,original,round_trip
0,6.4,6.4
1,10.0,10.0
2,2.3,2.3
3,13.6,13.6
4,0.7,0.7


---
## 6. Where This Fits in the Pipeline

Lesson 05 scaled the numeric columns; this lesson transformed them. In a real project, the **correct order is the other way around**: fix skewness with a transform first, *then* scale the transformed values. Scaling a still-skewed column (like Lesson 05 did directly) rescales the shape problem right along with it — it doesn't fix it. These two lessons taught the tools separately so each one was clear on its own, but a real cleaning pipeline runs:

**missing values → duplicates → outliers → encoding → transformation → scaling**

with transformation *before* the final scaling step.

---
## ⚠️ Common Mistakes

In [7]:
# -- Mistake 1: Using np.log() on a column that contains zero ------------------------------

broken = np.log(employees["years_experience"])
print("Mistake 1 — plain log() on a column with zeros silently produces -inf, not an error")
print("you'd notice right away:")
print(f"  -inf values produced: {np.isneginf(broken).sum()}")
print("  -> Use np.log1p() for any column that can contain 0.")

Mistake 1 — plain log() on a column with zeros silently produces -inf, not an error
you'd notice right away:
  -inf values produced: 1
  -> Use np.log1p() for any column that can contain 0.


/Users/aminuddin/Desktop/YTV/data-science/.venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [8]:
# -- Mistake 2: Transforming a column that's already roughly symmetric ---------------------

age_log = np.log(employees["age"])
print("Mistake 2 — transforming 'age' (skew already close to 0) doesn't just fail to help,")
print("it can make things WORSE by introducing skew in the other direction:")
print(f"  age skew before: {employees['age'].skew():.3f} (roughly symmetric)")
print(f"  age skew after log: {age_log.skew():.3f} (now noticeably LEFT-skewed)")
print("  -> Check .skew() first. Don't transform columns that aren't actually skewed.")

Mistake 2 — transforming 'age' (skew already close to 0) doesn't just fail to help,
it can make things WORSE by introducing skew in the other direction:
  age skew before: 0.119 (roughly symmetric)
  age skew after log: -0.294 (now noticeably LEFT-skewed)
  -> Check .skew() first. Don't transform columns that aren't actually skewed.


In [9]:
# -- Mistake 3: Forgetting to invert the transform before reporting a result in real units --

predicted_log_experience = 2.0  # pretend a model predicted this in log1p units

print("Mistake 3 — reporting a log-scale prediction as if it were already in years:")
print(f"  Reported directly (WRONG): '{predicted_log_experience} years of experience'")
print(f"  After np.expm1() (RIGHT):  '{np.expm1(predicted_log_experience):.1f} years of experience'")
print("  -> Any transform applied before modeling must be inverted before the result means anything.")

Mistake 3 — reporting a log-scale prediction as if it were already in years:
  Reported directly (WRONG): '2.0 years of experience'
  After np.expm1() (RIGHT):  '6.4 years of experience'
  -> Any transform applied before modeling must be inverted before the result means anything.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Skew Check

Load `data/employees_clean.csv` fresh. Print `.skew()` for every numeric column (`age`, `years_experience`, `salary`, `performance_band_encoded`). Which ones look like they'd benefit from a transform?

In [10]:
# Your code here

### Exercise 2 — Log Transform `salary`

Apply `np.log()` to `salary` (no zeros here, so plain `log` is safe) and print the skew before and after.

In [11]:
# Your code here

### Exercise 3 — log1p vs. log on `years_experience`

Apply both `np.log()` and `np.log1p()` to `years_experience`. Count how many `-inf` values plain `log` produces, and confirm `log1p` produces none.

In [12]:
# Your code here

### Exercise 4 — Round Trip a Transformed Column

Log1p-transform `years_experience`, then use `np.expm1()` to reverse it. Confirm the result matches the original column (within floating-point rounding) using `.round(6)` on both.

In [13]:
# Your code here

### Exercise 5 — (Challenge) Build the Full Pre-Scaling Pipeline

Write a function `transform_before_scaling(df)` that: log1p-transforms `years_experience`, log-transforms `salary`, and leaves `age` and `performance_band_encoded` untouched (since they're not meaningfully skewed). Return the modified DataFrame and print the new skew values to confirm the fix.

In [14]:
# Your code here

---
## 📌 Key Takeaways

- **`.skew()` tells you whether a column needs transforming before you reach for a plot or a guess** — near 0 is roughly symmetric, larger positive values mean a long right tail.
- **`np.log1p()` (not plain `np.log()`) is required for any column that can contain 0** — plain `log(0)` silently produces `-inf` and corrupts every downstream calculation without raising an error.
- **Transformation reduces skew, it doesn't erase genuine extreme values, and it belongs *before* scaling in a real pipeline** — fix the shape of a distribution first, then rescale the already-reshaped values.